In [2]:
%load_ext autoreload
%autoreload 2
%autosave 30
%matplotlib inline

Autosaving every 30 seconds


# Comparison and validation of FID implementation
As we used an autoencoder for the FID computation this notebook explores if this method can actually give insighs to the realness of the post-processed forecasts

For this, we will:
- Compute the FID for different post prosessing models
  - and explore relationships to the crps and other error measures
- Use different specification of the autoencoder
  - to explore how the size of the hidden dimension influences results
- Test if the FID increases when non-realistic forecasts are presented
  - for this we can use the averaged forecasts of the ensemble which should be "too smooth"
  - inject some sort of noise to the forecasts

In [3]:
import importlib
from itertools import product
from typing import Any

import hvplot.polars  # noqa: F401
import hydra
import lightning as L
import polars as pl
import torch
import xarray as xr
from einops import rearrange
from torch.utils.data import DataLoader, TensorDataset

from genpp import BASE_DIR
from genpp.data import FC_VARS, TEST_PREDICTIONS, TRAIN_PREDICTIONS, VAL_PREDICTIONS
from genpp.eval.FID import fid
from genpp.eval.utils import load_predictions_dataarray

## Load the encoder model

Best AutoEncoder: `feik/genpp/60kge09d`\
Best ClassifierEncoder: `feik/genpp/zln61d2q`

In [4]:
# Model ID is generated by WandB
RUN_PATH = "feik/genpp/zln61d2q"
MODEL_ID = RUN_PATH.split("/")[-1]
OUTPUT_DIR = BASE_DIR.parent.parent / "outputs"

MODEL_DIR = list(OUTPUT_DIR.rglob(f"*{MODEL_ID}*"))[0].parent.parent.parent
MODEL_CHECKPOINT = list(MODEL_DIR.rglob("*.ckpt"))[0]

with hydra.initialize_config_dir(config_dir=str(MODEL_DIR / ".hydra"), version_base=None):
    cfg = hydra.compose(
        config_name="config",
    )
class_path = cfg.model._target_

module_path, class_name = class_path.rsplit(".", 1)
module = importlib.import_module(module_path)
ModelClass = getattr(module, class_name)

encoder = ModelClass.load_from_checkpoint(MODEL_CHECKPOINT)

Ignored args: (), kwargs: {'use_rescaler': False, 'rescaler': None}
Loading channel_means from checkpoint
Loading channel_stds from checkpoint


## Generate the embeddings for the forecasts

We will compare these models:
- best emos model: `feik/genpp/k32mygar`
- best drn model: `feik/genpp/hn0gdrqm`
- best Chen model: `feik/genpp/qbuvhf5p`
- best FM model: `feik/genpp/blkpcik8`

In [5]:
models: dict[str, dict[str, Any]] = {
    "emos": {"id": "k32mygar"},
    "drn": {"id": "hn0gdrqm"},
    "chen": {"id": "qbuvhf5p"},
    "fm": {"id": "blkpcik8"},
}
# Val predictions

# Add the val predictions to the models dict
for model_name, model_info in models.items():
    model_dir = list(OUTPUT_DIR.rglob(f"*{model_info['id']}*"))[0].parent.parent.parent
    model_info["val_predictions"] = dict(
        (f.name, f) for f in model_dir.rglob("val_predictions*.zarr")
    )

In [6]:
trainer = L.Trainer(accelerator="auto", logger=False)

for _, info_dict in models.items():
    all_latents = {}
    for val_pred_name, val_pred_path in info_dict["val_predictions"].items():
        val_preds = load_predictions_dataarray(val_pred_path)
        # Convert to latents
        val_preds_tensor = torch.tensor(val_preds.to_numpy()).float()
        val_preds_tensor = rearrange(
            val_preds_tensor,
            "prediction sample channel height width -> (prediction sample) channel height width",
        )
        dataset = TensorDataset(val_preds_tensor)
        dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
        latents: list[torch.Tensor] = trainer.predict(encoder, dataloader)  # type: ignore
        latents_cat = torch.cat(latents, dim=0)
        all_latents[val_pred_name] = latents_cat
    info_dict["latents"] = all_latents

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


You are using a CUDA device ('NVIDIA RTX A5000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 2829/2829 [00:07<00:00, 367.41it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 2829/2829 [00:07<00:00, 369.52it/s]


...

Predicting DataLoader 0: 100%|██████████| 2829/2829 [00:06<00:00, 441.40it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 2829/2829 [00:06<00:00, 416.44it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 2829/2829 [00:05<00:00, 475.56it/s]


## Compute the latents for the ground truth dataset
This is the ifs ensemble predictions.

In [7]:
gt_path = BASE_DIR / "data" / "weatherbench2" / "ifs_ens.zarr"

splits = [("train", TRAIN_PREDICTIONS), ("val", VAL_PREDICTIONS), ("test", TEST_PREDICTIONS)]
gt_latents = {}
for split, split_prediction in splits:
    gt = (
        xr.open_zarr(gt_path)[FC_VARS]
        .to_dataarray("feature")
        .stack(prediction=("time", "prediction_timedelta"))
        .sel(prediction=split_prediction)
        .transpose("prediction", "number", "feature", "longitude", "latitude")
    )
    gt_tensor = torch.tensor(gt.to_numpy()).float()
    gt_tensor = rearrange(
        gt_tensor,
        "prediction number feature longitude latitude -> (prediction number) feature longitude latitude",
    )
    # Assert no NaNs
    assert not torch.isnan(gt_tensor).any()
    dataset = TensorDataset(gt_tensor)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
    latents: list[torch.Tensor] = trainer.predict(encoder, dataloader)  # type: ignore
    latents_cat = torch.cat(latents, dim=0)
    gt_latents[split] = latents_cat

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 8520/8520 [00:23<00:00, 362.46it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 2829/2829 [00:06<00:00, 416.22it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 2829/2829 [00:06<00:00, 454.90it/s]


In [8]:
rows = []
for (gt1_name, gt1_latent), (gt2_name, gt2_latent) in product(gt_latents.items(), repeat=2):
    if gt1_name > gt2_name:
        continue
    elif gt1_name == gt2_name:
        rows.append(
            {
                "gt_model": gt1_name,
                "model": gt2_name,
                "fid": 0.0,
            }
        )
    else:
        fid_value = fid(features1=gt1_latent, features2=gt2_latent)
        rows.append(
            {
                "gt_model": gt1_name,
                "model": gt2_name,
                "fid": fid_value,
            }
        )
        rows.append(
            {
                "gt_model": gt2_name,
                "model": gt1_name,
                "fid": fid_value,
            }
        )
for (gt_name, gt_latent), (model_type, model_dict) in product(gt_latents.items(), models.items()):
    for latent_name, latent_tensor in model_dict["latents"].items():
        extra_name = extra[-1].split(".")[0] if len(extra := latent_name.split("_")) == 3 else ""
        model_full_name = model_type + "_" + extra_name if extra_name else model_type
        fid_value = fid(features1=latent_tensor, features2=gt_latent)
        rows.append(
            {
                "gt_model": gt_name,
                "model": model_full_name,
                "fid": fid_value,
            }
        )

# Clip values to be non-negative as these come from numerical errors
fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

/home/feik/GenPP/src/genpp/eval/FID/__init__.py:126: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean: np.typing.NDArray[Any] = scipy.linalg.sqrtm(sigma1.mm(sigma2).numpy())  # pyright: ignore[reportAssignmentType]


gt_model,model,fid
str,str,f64
"""train""","""train""",0.0
"""train""","""val""",0.0
"""val""","""train""",0.0
"""val""","""val""",0.0
"""test""","""train""",0.0
…,…,…
"""test""","""emos_ecc""",1.30812
"""test""","""drn_gca""",14.374348
"""test""","""drn_ecc""",1.000534


## Notes on the following Plot

In the following we will compare the fid scores of the postprocessing models.\
Note that for all postprocessing models, the latents stem from the **validation set**.\
So the bad score for the other sets are somewhat expected.\
However the "real" forecasts should never have a worse score than the post-processed ones. This checks out and is a good sign.


In [ ]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

:HeatMap   [gt_model,model]   (fid)

## Lets add some intentionlly "bad" forecasts
For this we use the validation set of real forecasts and apply some transformations such as:
- gaussian noise
- gaussian blur
- black rectangles
- swirl
- salt and pepper noise

This list is taken from the FID paper: [GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium](https://arxiv.org/abs/1706.08500) (Heusel et al., 2018).

But first lets start with the mean forecast

In [10]:
# Load the data
from torch.utils.data import DataLoader, TensorDataset

from genpp.data import FC_VARS

ens_mean = xr.open_dataset(BASE_DIR / "data" / "weatherbench2" / "ens_flat_agg.zarr")
val_slice = slice("2021-01-01", "2021-12-31")

ens_mean = (
    ens_mean.sel(statistic="mean")[FC_VARS]
    .stack(prediction=["time", "prediction_timedelta"])
    .sel(prediction=VAL_PREDICTIONS)
    .to_dataarray("feature")
    .transpose("prediction", "feature", "longitude", "latitude")
)
mean_fc_tensor = torch.tensor(ens_mean.values).float()
mean_fc_dataset = TensorDataset(mean_fc_tensor)
mean_fc_dataloader = DataLoader(mean_fc_dataset, batch_size=128, shuffle=False)

trainer = L.Trainer(logger=False)
predictions = trainer.predict(encoder, mean_fc_dataloader)
predictions = torch.cat(predictions, dim=0)  # type: ignore

Trainer will use only 1 of 2 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=2)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 29/29 [00:00<00:00, 194.10it/s]


In [11]:
for gt_name, gt_model_latents in gt_latents.items():
    fid_value = fid(
        features1=gt_model_latents,
        features2=predictions,  # type: ignore
    )
    rows.append(
        {
            "gt_model": gt_name,
            "model": "mean_fc",
            "fid": fid_value,
        }
    )
# Clip values to be non-negative as these come from numerical errors
fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

/home/feik/GenPP/src/genpp/eval/FID/__init__.py:126: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean: np.typing.NDArray[Any] = scipy.linalg.sqrtm(sigma1.mm(sigma2).numpy())  # pyright: ignore[reportAssignmentType]


gt_model,model,fid
str,str,f64
"""train""","""train""",0.0
"""train""","""val""",0.0
"""val""","""train""",0.0
"""val""","""val""",0.0
"""test""","""train""",0.0
…,…,…
"""test""","""chen""",47.646622
"""test""","""fm""",7.978432
"""train""","""mean_fc""",23.815117


In [ ]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

:HeatMap   [gt_model,model]   (fid)

### Few notes on this

If the Autoencoder is good at encoding forecasts it is probably also good at encoding smoothed forecasts.\
Instead of using an autoencoder use some classifier that can differentiate between forecasts and too smooth forecasts (i.e. mean forecasts)


### Now add some intentional noise

In [13]:
from torchvision.transforms import v2

from genpp.data.zarr_dataset import TransformTensorDataset

In [14]:
ens_fc = (
    xr.open_dataset(BASE_DIR / "data" / "weatherbench2" / "ifs_ens.zarr")[FC_VARS]
    .stack(prediction=["time", "prediction_timedelta"])
    .sel(prediction=VAL_PREDICTIONS)
    .to_dataarray("feature")
    .transpose("prediction", "number", "feature", "longitude", "latitude")
)

In [15]:
gaussian_noise = v2.GaussianNoise(mean=0.0, sigma=1, clip=False)  # p=1 by default
gaussian_blur = v2.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0))  # p=1 by default
black_rectangle = v2.RandomErasing(
    p=1,
    scale=(0.02, 0.33),
    ratio=(0.3, 3.3),
    value=tuple(ens_fc.mean(["prediction", "number", "longitude", "latitude"]).values),  # type: ignore
)

defects = [gaussian_noise, gaussian_blur, black_rectangle]
for defect in defects:
    name = "ifs_ens+" + defect.__class__.__name__
    ens_fc_tensor = rearrange(
        torch.tensor(ens_fc.values).float(),
        "prediction number feature longitude latitude -> (prediction number) feature longitude latitude",
    )
    ens_fc_ds = TransformTensorDataset(ens_fc_tensor, transform=defect)
    dl = DataLoader(ens_fc_ds, batch_size=265, shuffle=False)
    predictions = trainer.predict(encoder, dl)
    predictions = torch.cat(predictions, dim=0)  # type: ignore
    for gt_name, gt_model_latents in gt_latents.items():
        fid_value = fid(
            features1=gt_model_latents,
            features2=predictions,  # type: ignore
        )
        rows.append(
            {
                "gt_model": gt_name,
                "model": name,
                "fid": fid_value,
            }
        )

fid_df = pl.DataFrame(rows).with_columns(pl.col("fid").clip(0, None))
fid_df

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 684/684 [00:29<00:00, 23.27it/s]


/home/feik/GenPP/src/genpp/eval/FID/__init__.py:126: LinAlgWarning: Matrix is singular. The result might be inaccurate or the array might not have a square root.
  covmean: np.typing.NDArray[Any] = scipy.linalg.sqrtm(sigma1.mm(sigma2).numpy())  # pyright: ignore[reportAssignmentType]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 684/684 [02:49<00:00,  4.04it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 684/684 [00:31<00:00, 21.59it/s]


gt_model,model,fid
str,str,f64
"""train""","""train""",0.0
"""train""","""val""",0.0
"""val""","""train""",0.0
"""val""","""val""",0.0
"""test""","""train""",0.0
…,…,…
"""val""","""ifs_ens+GaussianBlur""",11.462433
"""test""","""ifs_ens+GaussianBlur""",11.513041
"""train""","""ifs_ens+RandomErasing""",2.36893


In [ ]:
fid_df.hvplot.heatmap(  # pyright: ignore[reportAttributeAccessIssue]
    x="gt_model",
    y="model",
    C="fid",
    cmap="Reds",
    title="FID Scores Between Models",
    width=800,
    height=600,
    colorbar=True,
)

:HeatMap   [gt_model,model]   (fid)

## Compute the scores in latent space

In [17]:
import numpy as np

from genpp.data import OBSERVATIONS_FLAT_PATH
from genpp.models.loss import EnergyScore, EnsembleCRPS

In [18]:
# get the embeddings for the reforecast dataset
y_val = (
    xr.open_dataset(OBSERVATIONS_FLAT_PATH)
    .sel(
        time=VAL_PREDICTIONS.get_level_values("time")
        + VAL_PREDICTIONS.get_level_values("prediction_timedelta")
    )
    .to_dataarray("feature")
    .sel(feature=FC_VARS)
    .transpose("time", "feature", "longitude", "latitude")
    .rename({"time": "prediction_time"})
    .assign_coords(
        prediction=("prediction_time", VAL_PREDICTIONS),
    )
    .swap_dims({"prediction_time": "prediction"})
)

# pass through the encoder
y_tensor = torch.tensor(y_val.to_numpy()).float()
# Assert no NaNs
assert not torch.isnan(y_tensor).any()
dataset = TensorDataset(y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=False)
latents: list[torch.Tensor] = trainer.predict(encoder, dataloader)  # type: ignore
y_latents = torch.cat(latents, dim=0)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
/home/feik/GenPP/.pixi/envs/dev/lib/python3.13/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 57/57 [00:00<00:00, 485.11it/s]


In [35]:
scores = []
scores_t = []
scores_per_leadtime = []
CRPS = EnsembleCRPS(n_axis=-2)
ES = EnergyScore(clamp=False)

prediction_timedeltas = VAL_PREDICTIONS.get_level_values("prediction_timedelta").to_numpy()
td = np.unique(prediction_timedeltas)
td_h = [td / np.timedelta64(1, "h") for td in td]

for model_type, model_dict in models.items():
    for latent_name, latent_tensor in model_dict["latents"].items():
        extra_name = extra[-1].split(".")[0] if len(extra := latent_name.split("_")) == 3 else ""
        model_full_name = model_type + "_" + extra_name if extra_name else model_type
        # the CRPS expexts input dim [b, n, d] for x and [b, d] for y
        latent_reshaped = rearrange(latent_tensor, "(b n) d -> b n d", n=50)
        crps = CRPS(latent_reshaped, y_latents)
        energy_score = ES(latent_reshaped, y_latents)
        mse = torch.mean((latent_reshaped - rearrange(y_latents, "b d -> b 1 d")) ** 2, dim=-1)
        scores.append(
            {
                "model": model_full_name,
                "crps": crps.mean(),
                "energy_score": energy_score.mean(),
                "mse": mse.mean(),
            }
        )

        for delta, delta_h in zip(td, td_h):
            mask = prediction_timedeltas == delta
            crps_t = crps[mask].mean()
            energy_score_t = energy_score[mask].mean()
            mse_t = mse[mask].mean()
            scores_t.append(
                {
                    "model": model_full_name,
                    "prediction_timedelta": delta_h,
                    "crps": crps_t,
                    "energy_score": energy_score_t,
                    "mse": mse_t,
                }
            )

scores_df = pl.DataFrame(scores)
scores_t_df = pl.DataFrame(scores_t)

In [ ]:
p1 = scores_df.hvplot.bar(  # type: ignore[reportArgumentTypeMismatch]
    x="model",
    y="crps",
    title="CRPS on Val Set",
    width=400,
    height=400,
    rot=45,
)

p2 = scores_df.hvplot.bar(  # type: ignore[reportArgumentTypeMismatch]
    x="model",
    y="energy_score",
    title="Energy Score on Val Set",
    width=400,
    height=400,
    rot=45,
)

p3 = scores_df.hvplot.bar(  # type: ignore[reportArgumentTypeMismatch]
    x="model",
    y="mse",
    title="MSE on Val Set",
    width=400,
    height=400,
    rot=45,
)

(p1 + p2 + p3).cols(3)

:Layout
   .Bars.I   :Bars   [model]   (crps)
   .Bars.II  :Bars   [model]   (energy_score)
   .Bars.III :Bars   [model]   (mse)

In [43]:
p4 = scores_t_df.hvplot.line(  # type: ignore[reportArgumentTypeMismatch]
    x="prediction_timedelta",
    y="crps",
    by="model",
    title="CRPS on Val Set by Prediction Timedelta",
    width=400,
    height=500,
    legend="bottom",
    legend_cols=2,
)

p5 = scores_t_df.hvplot.line(  # type: ignore[reportArgumentTypeMismatch]
    x="prediction_timedelta",
    y="energy_score",
    by="model",
    title="Energy Score on Val Set by Prediction Timedelta",
    width=400,
    height=500,
    legend="bottom",
    legend_cols=2,
)

p6 = scores_t_df.hvplot.line(  # type: ignore[reportArgumentTypeMismatch]
    x="prediction_timedelta",
    y="mse",
    by="model",
    title="MSE on Val Set by Prediction Timedelta",
    width=400,
    height=500,
    legend="bottom",
    legend_cols=2,
)
(p4 + p5 + p6).cols(3)

:Layout
   .NdOverlay.I   :NdOverlay   [model]
      :Curve   [prediction_timedelta]   (crps)
   .NdOverlay.II  :NdOverlay   [model]
      :Curve   [prediction_timedelta]   (energy_score)
   .NdOverlay.III :NdOverlay   [model]
      :Curve   [prediction_timedelta]   (mse)